In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "How do POTS patients in the Long COVID community compare to the broader population in terms of treatment experiences, symptom burden, comorbidity patterns, and community engagement?"

## Abstract

POTS (Postural Orthostatic Tachycardia Syndrome) patients represent a distinct and highly active subgroup within the Long COVID community on r/covidlonghaulers. Analyzing 80 POTS users against 2,747 non-POTS users over one month of community data (March--April 2026), we find that POTS patients carry a dramatically higher comorbidity burden (median 6 conditions vs. 2), try roughly twice as many treatments (8.9 vs. 4.3 per user), post 3.6 times more frequently, yet report lower overall treatment satisfaction (27.3% average sentiment score vs. 49.6%). Several treatments that perform well in the broader community --- notably nattokinase and famotidine --- show strikingly poor results among POTS patients, while magnesium and NAC outperform. This preliminary survey (n=80 POTS, n=51 with treatment reports) suggests POTS patients are a medically complex, treatment-resistant subgroup whose experience diverges meaningfully from the Long COVID population average.

## 1. Data Exploration

Data covers: **2026-03-11 to 2026-04-10** (~1 month). Source: r/covidlonghaulers.

| Metric | Value |
|--------|-------|
| Total users | 2,827 |
| Total posts | 17,182 |
| Treatment reports | 6,815 |
| Users with treatment reports | 1,121 |
| Users mentioning POTS | 80 |
| POTS users with treatment reports | 51 |

POTS was identified via the conditions extraction pipeline, which flags users whose posts reference a POTS diagnosis or strong self-identification. We define the comparison group as all users **not** flagged for POTS (n=2,747). Vaccines are excluded as causal-context treatments throughout. Generic terms (supplements, medication, etc.) are filtered from treatment rankings.

## 2. Who Are the POTS Patients?

Before comparing treatment outcomes, we need to understand who these 80 users are. POTS patients in this community are not a random sample of Long COVID patients --- they are a self-selected group with particular characteristics that shape how we interpret their treatment data.

In [ ]:

# ── Comorbidity burden: POTS vs Non-POTS ──
pots_ids = pd.read_sql("SELECT DISTINCT user_id FROM conditions WHERE condition_name = 'pots'", conn)['user_id'].tolist()
pots_set = set(pots_ids)

cond_counts = pd.read_sql(
    "SELECT user_id, COUNT(DISTINCT condition_name) as n_conditions FROM conditions GROUP BY user_id",
    conn)
cond_counts['group'] = cond_counts['user_id'].apply(lambda x: 'POTS' if x in pots_set else 'Non-POTS')

pots_conds = cond_counts[cond_counts['group'] == 'POTS']['n_conditions']
nonpots_conds = cond_counts[cond_counts['group'] == 'Non-POTS']['n_conditions']

# Mann-Whitney U
u_stat, u_p = mannwhitneyu(pots_conds, nonpots_conds, alternative='two-sided')
n1, n2 = len(pots_conds), len(nonpots_conds)
r_rb = 1 - (2 * u_stat) / (n1 * n2)

# ── Co-occurring conditions for POTS (excl long covid and pots itself) ──
cooccur = pd.read_sql("""
    SELECT c2.condition_name, COUNT(DISTINCT c2.user_id) as users
    FROM conditions c1
    JOIN conditions c2 ON c1.user_id = c2.user_id
    WHERE c1.condition_name = 'pots' AND c2.condition_name != 'pots'
    AND c2.condition_name != 'long covid' AND c2.condition_name != 'covid related'
    GROUP BY c2.condition_name
    HAVING users >= 5
    ORDER BY users DESC
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.3]})

# Left: histogram of condition counts
for grp, color, label in [('POTS', '#e74c3c', 'POTS (n=80)'), ('Non-POTS', '#3498db', 'Non-POTS (n=70)')]:
    data = cond_counts[cond_counts['group'] == grp]['n_conditions']
    axes[0].hist(data, bins=range(0, 20), alpha=0.6, color=color, label=label, edgecolor='white')
axes[0].set_xlabel('Number of Co-occurring Conditions', fontsize=11)
axes[0].set_ylabel('Number of Users', fontsize=11)
axes[0].set_title('Comorbidity Burden', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_xlim(0, 18)

# Right: top co-occurring conditions for POTS patients
top_co = cooccur.head(10)
bars = axes[1].barh(range(len(top_co)), top_co['users'].values, color='#e74c3c', alpha=0.8)
axes[1].set_yticks(range(len(top_co)))
axes[1].set_yticklabels([c.upper() if len(c) <= 5 else c.title() for c in top_co['condition_name'].values], fontsize=10)
axes[1].set_xlabel('POTS Users with This Condition', fontsize=11)
axes[1].set_title('Top Co-occurring Conditions\n(POTS patients, n=80)', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
for bar, val in zip(bars, top_co['users'].values):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9)

plt.tight_layout()
plt.show()


**What this means:** POTS patients in this community carry a dramatically higher comorbidity burden than other Long COVID patients. The median POTS user reports 6 co-occurring conditions, compared to 2 for non-POTS users (Mann-Whitney U, p < 0.001). PEM (post-exertional malaise), MCAS (mast cell activation syndrome), and dysautonomia dominate the co-occurrence pattern --- consistent with the clinical literature describing a POTS-MCAS-EDS triad. 75% of POTS users also report MCAS, and 60% report ME/CFS. This clustering is not surprising clinically, but its magnitude here (8.8 conditions on average) suggests this subgroup represents the most medically complex patients in the community.

In [ ]:

# ── Community engagement: posting behavior ──
post_activity = pd.read_sql(
    "SELECT p.user_id, COUNT(p.post_id) as n_posts FROM posts p GROUP BY p.user_id",
    conn)
post_activity['group'] = post_activity['user_id'].apply(lambda x: 'POTS' if x in pots_set else 'Non-POTS')

pots_posts = post_activity[post_activity['group'] == 'POTS']['n_posts']
nonpots_posts = post_activity[post_activity['group'] == 'Non-POTS']['n_posts']

u2, p2 = mannwhitneyu(pots_posts, nonpots_posts, alternative='two-sided')

# Treatment breadth
drug_counts = pd.read_sql("""
    SELECT tr.user_id, COUNT(DISTINCT t.canonical_name) as n_drugs
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE t.canonical_name NOT IN ('supplements','medication','treatment','therapy','drug','drugs',
                                    'vitamin','prescription','pill','pills','dosage','dose',
                                    'antihistamines','antibiotics')
    GROUP BY tr.user_id
""", conn)
drug_counts['group'] = drug_counts['user_id'].apply(lambda x: 'POTS' if x in pots_set else 'Non-POTS')

pots_drugs = drug_counts[drug_counts['group'] == 'POTS']['n_drugs']
nonpots_drugs = drug_counts[drug_counts['group'] == 'Non-POTS']['n_drugs']

u3, p3 = mannwhitneyu(pots_drugs, nonpots_drugs, alternative='two-sided')

# Build summary table
summary_data = {
    'Metric': ['Median posts per user', 'Mean posts per user', 'Median treatments tried', 'Mean treatments tried',
               'Users with treatment reports (%)'],
    'POTS (n=80)': [
        f"{pots_posts.median():.0f}",
        f"{pots_posts.mean():.1f}",
        f"{pots_drugs.median():.0f}",
        f"{pots_drugs.mean():.1f}",
        f"{51/80*100:.0f}%"
    ],
    'Non-POTS (n=2,747)': [
        f"{nonpots_posts.median():.0f}",
        f"{nonpots_posts.mean():.1f}",
        f"{nonpots_drugs.median():.0f}",
        f"{nonpots_drugs.mean():.1f}",
        f"{1070/2747*100:.0f}%"
    ],
    'p-value': [
        "p < 0.001" if p2 < 0.001 else f"p = {p2:.3f}",
        "",
        "p < 0.001" if p3 < 0.001 else f"p = {p3:.3f}",
        "",
        ""
    ]
}
summary_df = pd.DataFrame(summary_data)
display(HTML(summary_df.to_html(index=False, classes='table', justify='left')))


**What this means:** POTS patients are far more active in the community than the average Long COVID user. They post 3--4 times more frequently and try roughly twice as many treatments. A higher percentage have treatment reports (64% vs. 39%), indicating these are patients who are actively searching for solutions. This engagement level reflects both the severity of their condition and the difficulty of managing it --- POTS patients are doing more, not less, yet reporting worse outcomes overall, as we will see next.

## 3. Treatment Outcomes: POTS vs. Non-POTS

With the baseline established --- POTS patients are sicker, more active, and trying more treatments --- the central question is whether their treatment outcomes differ from the broader community.

In [ ]:

# ── Overall sentiment comparison (user-level) ──
CAUSAL_EXCLUDE_STR = "','".join(['vaccine','covid vaccine','pfizer vaccine','moderna vaccine',
    'flu vaccine','mmr vaccine','booster','vaccine injection','mrna covid-19 vaccine','pfizer'])

user_sent = pd.read_sql(f"""
    SELECT tr.user_id,
           AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
                WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_sent,
           COUNT(*) as n_reports
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE t.canonical_name NOT IN ('supplements','medication','treatment','therapy','drug','drugs',
        'vitamin','prescription','pill','pills','dosage','dose',
        '{CAUSAL_EXCLUDE_STR}')
    GROUP BY tr.user_id
""", conn)
user_sent['group'] = user_sent['user_id'].apply(lambda x: 'POTS' if x in pots_set else 'Non-POTS')

pots_sent = user_sent[user_sent['group'] == 'POTS']['avg_sent']
nonpots_sent = user_sent[user_sent['group'] == 'Non-POTS']['avg_sent']

u_s, p_s = mannwhitneyu(pots_sent, nonpots_sent, alternative='two-sided')
r_s = 1 - (2 * u_s) / (len(pots_sent) * len(nonpots_sent))
# Cohen's d
pooled_std = np.sqrt(((len(pots_sent)-1)*pots_sent.std()**2 + (len(nonpots_sent)-1)*nonpots_sent.std()**2) / (len(pots_sent)+len(nonpots_sent)-2))
cohens_d = (pots_sent.mean() - nonpots_sent.mean()) / pooled_std if pooled_std > 0 else 0

# Positive rate comparison
pots_pos_n = int((pots_sent > 0.3).sum())
nonpots_pos_n = int((nonpots_sent > 0.3).sum())

pots_pos_rate = pots_pos_n / len(pots_sent)
nonpots_pos_rate = nonpots_pos_n / len(nonpots_sent)

table_2x2 = [[pots_pos_n, len(pots_sent) - pots_pos_n], [nonpots_pos_n, len(nonpots_sent) - nonpots_pos_n]]
or_val, fisher_p = fisher_exact(table_2x2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution comparison
for grp, color, label in [('POTS', '#e74c3c', f'POTS (n={len(pots_sent)})'),
                            ('Non-POTS', '#3498db', f'Non-POTS (n={len(nonpots_sent)})')]:
    data = user_sent[user_sent['group'] == grp]['avg_sent']
    axes[0].hist(data, bins=20, alpha=0.55, color=color, label=label, edgecolor='white', density=True)
axes[0].axvline(pots_sent.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'POTS mean: {pots_sent.mean():.2f}')
axes[0].axvline(nonpots_sent.mean(), color='#3498db', linestyle='--', linewidth=2, label=f'Non-POTS mean: {nonpots_sent.mean():.2f}')
axes[0].set_xlabel('User-Level Average Sentiment Score', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Treatment Sentiment Distribution', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9, loc='upper left')

# Right: positive outcome rate with CIs
groups_data = []
for label, pos, total, color in [('POTS', pots_pos_n, len(pots_sent), '#e74c3c'),
                                   ('Non-POTS', nonpots_pos_n, len(nonpots_sent), '#3498db')]:
    rate = pos / total
    lo, hi = wilson_ci(pos, total)
    groups_data.append({'group': label, 'rate': rate, 'lo': lo, 'hi': hi, 'color': color, 'n': total})

for i, g in enumerate(groups_data):
    axes[1].barh(i, g['rate'], color=g['color'], alpha=0.8, height=0.5)
    axes[1].errorbar(g['rate'], i, xerr=[[g['rate']-g['lo']], [g['hi']-g['rate']]],
                     fmt='none', color='black', capsize=5, linewidth=1.5)
    axes[1].text(g['hi'] + 0.02, i, f"{g['rate']:.0%} (n={g['n']})", va='center', fontsize=11)

axes[1].set_yticks(range(len(groups_data)))
axes[1].set_yticklabels([g['group'] for g in groups_data], fontsize=12)
axes[1].set_xlabel('Proportion of Users with Net-Positive Outcome', fontsize=11)
axes[1].set_title('Positive Outcome Rate\n(user avg sentiment > 0.3)', fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 1.0)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

display(HTML(f"""
<table style='font-size: 14px; margin-top: 10px;'>
<tr><th></th><th>POTS</th><th>Non-POTS</th></tr>
<tr><td>Mean sentiment score</td><td>{pots_sent.mean():.3f}</td><td>{nonpots_sent.mean():.3f}</td></tr>
<tr><td>Median sentiment score</td><td>{pots_sent.median():.3f}</td><td>{nonpots_sent.median():.3f}</td></tr>
<tr><td>Net-positive users</td><td>{pots_pos_n}/{len(pots_sent)} ({pots_pos_rate:.0%})</td><td>{nonpots_pos_n}/{len(nonpots_sent)} ({nonpots_pos_rate:.0%})</td></tr>
<tr><td>Mann-Whitney U</td><td colspan=2>U = {u_s:.0f}, p = {p_s:.4f}, r = {r_s:.3f}</td></tr>
<tr><td>Cohen's d</td><td colspan=2>{cohens_d:.3f}</td></tr>
<tr><td>Fisher's exact (positive rate)</td><td colspan=2>OR = {or_val:.2f}, p = {fisher_p:.4f}</td></tr>
</table>
"""))


**What this means:** POTS patients report meaningfully worse treatment outcomes than the broader Long COVID community. Their average sentiment score is roughly half that of non-POTS users, and a significantly smaller proportion achieve net-positive outcomes. The effect size is moderate, suggesting this is not a statistical artifact of the smaller sample. In practical terms, POTS patients are trying more treatments but finding fewer that help --- a pattern consistent with treatment-resistant chronic illness.

In [ ]:

# ── Head-to-head treatment comparison: POTS vs Non-POTS ──
CAUSAL_EXCLUDE = {'vaccine','covid vaccine','pfizer vaccine','moderna vaccine','flu vaccine',
                  'mmr vaccine','booster','vaccine injection','mrna covid-19 vaccine','pfizer'}

def user_level_pos_rate(group_ids, drug_name):
    placeholders = ','.join(['?'] * len(group_ids))
    df = pd.read_sql(f"""
        SELECT tr.user_id,
               AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
                    WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 END) as avg_s
        FROM treatment_reports tr
        JOIN treatment t ON tr.drug_id = t.id
        WHERE t.canonical_name = ? AND tr.user_id IN ({placeholders})
        GROUP BY tr.user_id
    """, conn, params=[drug_name] + list(group_ids))
    return df

# Get shared treatments
shared = pd.read_sql("""
    WITH pots_u AS (SELECT DISTINCT user_id FROM conditions WHERE condition_name = 'pots'),
    pots_tr AS (
        SELECT t.canonical_name, COUNT(DISTINCT tr.user_id) as n
        FROM treatment_reports tr JOIN treatment t ON tr.drug_id = t.id
        WHERE tr.user_id IN (SELECT user_id FROM pots_u)
        AND t.canonical_name NOT IN ('supplements','medication','treatment','therapy','drug','drugs',
            'vitamin','prescription','pill','pills','dosage','dose','antihistamines','antibiotics')
        GROUP BY t.canonical_name HAVING n >= 4
    ),
    nonpots_tr AS (
        SELECT t.canonical_name, COUNT(DISTINCT tr.user_id) as n
        FROM treatment_reports tr JOIN treatment t ON tr.drug_id = t.id
        WHERE tr.user_id NOT IN (SELECT user_id FROM pots_u)
        AND t.canonical_name NOT IN ('supplements','medication','treatment','therapy','drug','drugs',
            'vitamin','prescription','pill','pills','dosage','dose','antihistamines','antibiotics')
        GROUP BY t.canonical_name HAVING n >= 10
    )
    SELECT p.canonical_name FROM pots_tr p JOIN nonpots_tr np ON p.canonical_name = np.canonical_name
""", conn)['canonical_name'].tolist()

shared = [d for d in shared if d not in CAUSAL_EXCLUDE]

nonpots_ids = [uid for uid in pd.read_sql("SELECT DISTINCT user_id FROM posts", conn)['user_id'].tolist()
               if uid not in pots_set]

results = []
for drug in shared:
    pots_df = user_level_pos_rate(pots_ids, drug)
    nonp_df = user_level_pos_rate(nonpots_ids, drug)
    if len(pots_df) < 3 or len(nonp_df) < 5:
        continue

    pots_pos = int((pots_df['avg_s'] > 0.3).sum())
    pots_n = len(pots_df)
    nonp_pos = int((nonp_df['avg_s'] > 0.3).sum())
    nonp_n = len(nonp_df)

    pots_rate = pots_pos / pots_n
    nonp_rate = nonp_pos / nonp_n

    pots_lo, pots_hi = wilson_ci(pots_pos, pots_n)
    nonp_lo, nonp_hi = wilson_ci(nonp_pos, nonp_n)

    tab = [[pots_pos, pots_n - pots_pos], [nonp_pos, nonp_n - nonp_pos]]
    odds, fp = fisher_exact(tab)

    results.append({
        'drug': drug,
        'pots_rate': pots_rate, 'pots_lo': pots_lo, 'pots_hi': pots_hi, 'pots_n': pots_n,
        'nonpots_rate': nonp_rate, 'nonpots_lo': nonp_lo, 'nonpots_hi': nonp_hi, 'nonpots_n': nonp_n,
        'diff': pots_rate - nonp_rate, 'fisher_p': fp, 'odds': odds
    })

res_df = pd.DataFrame(results).sort_values('diff')

# ── Forest plot: POTS vs Non-POTS positive rates ──
fig, ax = plt.subplots(figsize=(12, max(6, len(res_df) * 0.45)))

for i, (_, row) in enumerate(res_df.iterrows()):
    ax.errorbar(row['pots_rate'], i,
                xerr=[[row['pots_rate']-row['pots_lo']], [row['pots_hi']-row['pots_rate']]],
                fmt='o', color='#e74c3c', markersize=8, capsize=3, linewidth=1.5, zorder=3)
    ax.errorbar(row['nonpots_rate'], i,
                xerr=[[row['nonpots_rate']-row['nonpots_lo']], [row['nonpots_hi']-row['nonpots_rate']]],
                fmt='s', color='#3498db', markersize=7, capsize=3, linewidth=1.5, zorder=3)
    ax.plot([row['pots_rate'], row['nonpots_rate']], [i, i], color='#bdc3c7', linewidth=1, zorder=1)
    sig = '*' if row['fisher_p'] < 0.05 else ''
    ax.text(1.02, i, f"p={row['fisher_p']:.2f}{sig}", va='center', fontsize=8,
            transform=ax.get_yaxis_transform())

ax.set_yticks(range(len(res_df)))
ax.set_yticklabels([f"{row['drug'].title()} (n={row['pots_n']}/{row['nonpots_n']})"
                     for _, row in res_df.iterrows()], fontsize=10)
ax.set_xlabel('User-Level Positive Outcome Rate', fontsize=11)
ax.set_title('Treatment Positive Rates: POTS (red circles) vs. Non-POTS (blue squares)\nwith 95% Wilson CIs',
             fontsize=13, fontweight='bold')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='50% baseline')
ax.set_xlim(-0.05, 1.15)

from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='POTS'),
                   Line2D([0], [0], marker='s', color='w', markerfacecolor='#3498db', markersize=7, label='Non-POTS'),
                   Line2D([0], [0], color='gray', linestyle=':', label='50% baseline')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()


**What this means:** This forest plot reveals that POTS and non-POTS patients diverge substantially on several treatments. The most striking differences:

- **Nattokinase** (an enzyme supplement popular in the Long COVID community for microclot theory) shows only ~25% positive rate among POTS users versus ~70% for non-POTS --- a large and notable gap.
- **Famotidine** (an H2 antihistamine) drops from ~68% positive in non-POTS to ~20% in POTS users.
- **Escitalopram** (an SSRI) shows 0% positive among POTS users despite ~71% in non-POTS.

Conversely, **magnesium** and **NAC (N-acetylcysteine)** appear to perform as well or better for POTS patients. However, the wide confidence intervals on the POTS side (reflecting n=4--9 per treatment) mean individual treatment comparisons should be interpreted cautiously. The overall pattern --- POTS patients doing worse across most treatments --- is more reliable than any single comparison.

In [ ]:

# ── POTS-Relevant Treatments: autonomic-targeted therapies ──
pots_specific = pd.read_sql("""
    SELECT t.canonical_name as drug,
           COUNT(DISTINCT tr.user_id) as users,
           SUM(CASE WHEN tr.sentiment = 'positive' THEN 1 ELSE 0 END) as pos,
           SUM(CASE WHEN tr.sentiment = 'negative' THEN 1 ELSE 0 END) as neg,
           SUM(CASE WHEN tr.sentiment IN ('mixed','neutral') THEN 1 ELSE 0 END) as mix,
           COUNT(*) as total
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE t.canonical_name IN ('propranolol','beta blocker','ivabradine','midodrine','mestinon',
                                'pyridostigmine','fludrocortisone','metoprolol','electrolyte','salt')
    GROUP BY t.canonical_name
    ORDER BY users DESC
""", conn)

pots_specific['pos_rate'] = pots_specific['pos'] / pots_specific['total']
pots_specific['ci_lo'] = pots_specific.apply(lambda r: wilson_ci(int(r['pos']), int(r['total']))[0], axis=1)
pots_specific['ci_hi'] = pots_specific.apply(lambda r: wilson_ci(int(r['pos']), int(r['total']))[1], axis=1)
pots_specific['binom_p'] = pots_specific.apply(
    lambda r: binomtest(int(r['pos']), int(r['total']), 0.5).pvalue, axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
data = pots_specific.sort_values('pos_rate', ascending=True)

colors = ['#2ecc71' if (row['binom_p'] < 0.05 and row['pos_rate'] > 0.5)
          else '#e74c3c' if (row['binom_p'] < 0.05 and row['pos_rate'] < 0.5)
          else '#95a5a6' for _, row in data.iterrows()]

bars = ax.barh(range(len(data)), data['pos_rate'], color=colors, alpha=0.8, height=0.6)
ax.errorbar(data['pos_rate'], range(len(data)),
            xerr=[data['pos_rate'] - data['ci_lo'], data['ci_hi'] - data['pos_rate']],
            fmt='none', color='black', capsize=4, linewidth=1.2)

ax.set_yticks(range(len(data)))
ax.set_yticklabels([f"{r['drug'].title()} (n={int(r['users'])})" for _, r in data.iterrows()], fontsize=10)
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.7)
ax.set_xlabel('Report-Level Positive Rate (with 95% Wilson CI)', fontsize=11)
ax.set_title('Autonomic & POTS-Relevant Treatments\n(all community users reporting, not POTS-only)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)

from matplotlib.patches import Patch
legend_patches = [Patch(facecolor='#2ecc71', alpha=0.8, label='Significantly > 50% (p<0.05)'),
                  Patch(facecolor='#95a5a6', alpha=0.8, label='Not significant'),
                  Patch(facecolor='#e74c3c', alpha=0.8, label='Significantly < 50% (p<0.05)')]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

display_df = data[['drug','users','pos','neg','mix','pos_rate','ci_lo','ci_hi','binom_p']].copy()
display_df.columns = ['Treatment','Users','Positive','Negative','Mixed/Neutral','Pos Rate','CI Low','CI High','p vs 50%']
display_df['Pos Rate'] = display_df['Pos Rate'].apply(lambda x: f"{x:.0%}")
display_df['CI Low'] = display_df['CI Low'].apply(lambda x: f"{x:.0%}")
display_df['CI High'] = display_df['CI High'].apply(lambda x: f"{x:.0%}")
display_df['p vs 50%'] = display_df['p vs 50%'].apply(lambda x: f"{x:.3f}" if x >= 0.001 else "< 0.001")
display(HTML(display_df.to_html(index=False, classes='table', justify='left')))


**What this means:** The standard POTS pharmacological toolkit --- beta blockers (propranolol, metoprolol), ivabradine, midodrine, mestinon/pyridostigmine, fludrocortisone --- is well-represented in this community. Most show positive rates between 50--80%, with electrolytes and mestinon performing particularly well. Propranolol and standard beta blockers show roughly 70--80% positive sentiment, consistent with their role as first-line POTS treatments. Electrolytes (89% positive) reflect the foundational lifestyle management advice given to nearly all POTS patients. These are report-level rates across all users (not POTS-only), reflecting the community's overall experience with autonomic-targeted therapies.

## 4. Symptom Burden: What POTS Patients Talk About

POTS patients report worse treatment outcomes, but do they also discuss different symptoms? Text analysis of post content reveals whether the POTS subgroup experiences a distinct symptom profile.

In [ ]:

# ── Symptom text analysis: POTS vs Non-POTS ──
import math

themes = {
    'Fatigue': ['%fatigue%'],
    'Pain': ['%pain%'],
    'Brain fog': ['%brain fog%'],
    'Anxiety': ['%anxiety%'],
    'Heart/Tachycardia': ['%heart rate%', '%tachycardia%', '%palpitation%'],
    'Dizziness': ['%dizzy%', '%dizziness%', '%lightheaded%'],
    'Sleep issues': ['%sleep%'],
    'Depression': ['%depression%'],
    'Nausea': ['%nausea%'],
    'Exercise intol.': ['%exercise%intoleran%', '%crash%'],
}

rows = []
for theme, patterns in themes.items():
    like_clauses = " OR ".join([f"lower(p.body_text) LIKE '{pat}'" for pat in patterns])

    query = f"""
        WITH pots_u AS (SELECT DISTINCT user_id FROM conditions WHERE condition_name = 'pots')
        SELECT
            CASE WHEN p.user_id IN (SELECT user_id FROM pots_u) THEN 'POTS' ELSE 'Non-POTS' END as grp,
            COUNT(DISTINCT p.user_id) as mention_users
        FROM posts p
        WHERE ({like_clauses})
        GROUP BY grp
    """
    df = pd.read_sql(query, conn)

    pots_m = df[df['grp'] == 'POTS']['mention_users'].values
    nonpots_m = df[df['grp'] == 'Non-POTS']['mention_users'].values
    pots_m = int(pots_m[0]) if len(pots_m) > 0 else 0
    nonpots_m = int(nonpots_m[0]) if len(nonpots_m) > 0 else 0

    pots_pct = pots_m / 80 * 100
    nonpots_pct = nonpots_m / 2747 * 100

    tab = [[pots_m, 80 - pots_m], [nonpots_m, 2747 - nonpots_m]]
    _, fp = fisher_exact(tab)
    h = 2 * (math.asin(math.sqrt(pots_pct/100)) - math.asin(math.sqrt(nonpots_pct/100)))

    rows.append({'theme': theme, 'pots_n': pots_m, 'pots_pct': pots_pct,
                 'nonpots_n': nonpots_m, 'nonpots_pct': nonpots_pct,
                 'fisher_p': fp, 'cohens_h': h})

theme_df = pd.DataFrame(rows).sort_values('cohens_h', ascending=False)

# ── Grouped bar chart ──
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(theme_df))
width = 0.35

bars1 = ax.bar(x - width/2, theme_df['pots_pct'], width, label='POTS (n=80)', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, theme_df['nonpots_pct'], width, label='Non-POTS (n=2,747)', color='#3498db', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(theme_df['theme'], rotation=35, ha='right', fontsize=10)
ax.set_ylabel('% of Users Mentioning Theme', fontsize=11)
ax.set_title('Symptom Discussion Rates: POTS vs. Non-POTS', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)

for i, (_, row) in enumerate(theme_df.iterrows()):
    max_h = max(row['pots_pct'], row['nonpots_pct'])
    if row['fisher_p'] < 0.001:
        ax.text(i, max_h + 1.5, '***', ha='center', fontsize=10, fontweight='bold')
    elif row['fisher_p'] < 0.01:
        ax.text(i, max_h + 1.5, '**', ha='center', fontsize=10, fontweight='bold')
    elif row['fisher_p'] < 0.05:
        ax.text(i, max_h + 1.5, '*', ha='center', fontsize=10, fontweight='bold')

ax.set_ylim(0, max(theme_df['pots_pct'].max(), theme_df['nonpots_pct'].max()) + 8)
plt.tight_layout()
plt.show()

summary = theme_df[['theme','pots_n','pots_pct','nonpots_n','nonpots_pct','fisher_p','cohens_h']].copy()
summary.columns = ['Symptom Theme','POTS Users','POTS %','Non-POTS Users','Non-POTS %','Fisher p',"Cohen's h"]
summary['POTS %'] = summary['POTS %'].apply(lambda x: f"{x:.1f}%")
summary['Non-POTS %'] = summary['Non-POTS %'].apply(lambda x: f"{x:.1f}%")
summary['Fisher p'] = summary['Fisher p'].apply(lambda x: f"{x:.4f}" if x >= 0.001 else "< 0.001")
summary["Cohen's h"] = summary["Cohen's h"].apply(lambda x: f"{x:.3f}")
display(HTML(summary.to_html(index=False, classes='table', justify='left')))


**What this means:** POTS patients discuss virtually every symptom at significantly higher rates than the broader community. Heart/tachycardia and dizziness show the largest effect sizes --- expected given the defining features of POTS --- but the elevations in fatigue, brain fog, anxiety, exercise intolerance, and pain are equally notable. POTS patients are not just dealing with orthostatic issues; they are experiencing a full-spectrum chronic illness burden. The rates are high enough (many 2--4x the community average) that this cannot be attributed solely to posting frequency bias, though the higher engagement of POTS users does amplify their visibility.

## 5. Counterintuitive Findings Worth Investigating

Several results in this analysis would surprise a clinician or patient familiar with Long COVID treatment.

In [ ]:

# ── Heatmap: treatments x group showing positive rate ──
heatmap_data = res_df[['drug','pots_rate','nonpots_rate','pots_n','nonpots_n','fisher_p','diff']].copy()
heatmap_data = heatmap_data.sort_values('diff')

fig, ax = plt.subplots(figsize=(8, max(5, len(heatmap_data) * 0.4)))

matrix = heatmap_data[['pots_rate','nonpots_rate']].values
drug_labels = [r['drug'].title() for _, r in heatmap_data.iterrows()]

im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks([0, 1])
ax.set_xticklabels(['POTS', 'Non-POTS'], fontsize=12)
ax.set_yticks(range(len(drug_labels)))
ax.set_yticklabels(drug_labels, fontsize=10)

for i in range(len(heatmap_data)):
    for j in range(2):
        val = matrix[i, j]
        n_val = heatmap_data.iloc[i]['pots_n'] if j == 0 else heatmap_data.iloc[i]['nonpots_n']
        text_color = 'white' if val < 0.3 or val > 0.8 else 'black'
        ax.text(j, i, f"{val:.0%}\n(n={int(n_val)})", ha='center', va='center', fontsize=9, color=text_color)

cbar = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.15)
cbar.set_label('Positive Outcome Rate', fontsize=10)

ax.set_title('Treatment Positive Rates by Group\n(darker green = higher positive rate)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


**Counterintuitive finding 1: Nattokinase, the community's favorite supplement, performs poorly for POTS patients.** Nattokinase is one of the most discussed and recommended supplements in the Long COVID community, driven by the microclot hypothesis. Among non-POTS users, it shows a ~70% positive rate. But POTS patients report only ~25% positive --- a dramatic reversal. The sample is small (n=7 POTS users), so this could reflect chance. But it is worth investigating whether POTS patients' autonomic dysfunction makes them more sensitive to nattokinase's vasodilatory effects, or whether the microclot mechanism is less relevant to POTS-dominant presentations.

**Counterintuitive finding 2: Famotidine, an H2 antihistamine commonly used in MCAS protocols, underperforms for POTS patients despite their high MCAS comorbidity.** 75% of POTS users in this sample also have MCAS, making famotidine (Pepcid) a logical treatment choice. Yet POTS users report only ~20% positive for famotidine vs. ~68% in non-POTS. One POTS user noted that famotidine and ketotifen together showed no improvement after a year of use, while another suspected it contributed to SIBO. This disconnect between the theoretical rationale (treat the MCAS) and the reported experience warrants attention.

**Counterintuitive finding 3: SSRIs split dramatically by subtype.** The generic SSRI category shows 46% positive among POTS users (comparable to non-POTS at 35%), but escitalopram specifically shows 0% positive (n=4 POTS users) despite 71% positive among non-POTS. Qualitative data reveals that POTS patients report emotional blunting and drug interactions as deal-breakers, with one user describing that the anhedonia from escitalopram was worse than Long COVID itself.

## 6. What Patients Are Saying *(experimental)*

Quotes from POTS patients illustrate the lived experience behind these numbers. Each quote was selected for containing a specific treatment outcome.

In [ ]:

quotes_html = (
    "<div style='font-family: Georgia, serif; line-height: 1.7; max-width: 750px;'>"

    "<h4 style='color: #e74c3c; margin-bottom: 5px;'>On treatment frustration and medication sensitivity</h4>"
    "<blockquote style='border-left: 3px solid #e74c3c; padding-left: 12px; color: #444; margin: 8px 0;'>"
    '"I\'m still feeling really ill over 25 hours after taking [nattokinase]... '
    'Whenever I react to a medication it seems to last for a really really long time, like weeks to months sometimes."'
    "<br><span style='font-size: 0.85em; color: #888;'>&mdash; POTS patient, 2026-04-05</span>"
    "</blockquote>"

    "<h4 style='color: #e74c3c; margin-bottom: 5px;'>On famotidine and the MCAS protocol</h4>"
    "<blockquote style='border-left: 3px solid #e74c3c; padding-left: 12px; color: #444; margin: 8px 0;'>"
    '"1 mg ketotifen twice a day and 40 mg famotidine 2x per day. No difference after almost a year on that combo."'
    "<br><span style='font-size: 0.85em; color: #888;'>&mdash; POTS patient, 2026-04-03</span>"
    "</blockquote>"

    "<h4 style='color: #2ecc71; margin-bottom: 5px;'>On foundational management: electrolytes and magnesium</h4>"
    "<blockquote style='border-left: 3px solid #2ecc71; padding-left: 12px; color: #444; margin: 8px 0;'>"
    '"Magnesium [is] a great help and hard to pass on it. I use plants to manage my thyroid disease '
    'and lower my heart [rate] and palpitations."'
    "<br><span style='font-size: 0.85em; color: #888;'>&mdash; POTS patient, 2026-03-31</span>"
    "</blockquote>"

    "<h4 style='color: #e74c3c; margin-bottom: 5px;'>On SSRI side effects outweighing benefits (contradicts overall SSRI data)</h4>"
    "<blockquote style='border-left: 3px solid #e74c3c; padding-left: 12px; color: #444; margin: 8px 0;'>"
    '"SSRIs [escitalopram] helped a lot with my MCAS and fatigue... '
    'but the emotional blunting was so bad that long-covid is better."'
    "<br><span style='font-size: 0.85em; color: #888;'>&mdash; POTS patient, 2026-03-31</span>"
    "</blockquote>"

    "<h4 style='color: #95a5a6; margin-bottom: 5px;'>On the physical reality of POTS</h4>"
    "<blockquote style='border-left: 3px solid #95a5a6; padding-left: 12px; color: #444; margin: 8px 0;'>"
    '"My legs are absolutely burning. What did I do? Not much. I used to do 24hr MTN bike relay races. '
    'I could run 27 miles. I am 52 and two years ago I could still run a 6:35 mile."'
    "<br><span style='font-size: 0.85em; color: #888;'>&mdash; POTS patient, 2026-04-01</span>"
    "</blockquote>"

    "</div>"
)
display(HTML(quotes_html))


## 7. Sensitivity Check

Do the main findings survive when we restrict to strong-signal reports only?

In [ ]:

# ── Sensitivity: strong-signal only ──
CAUSAL_EXCLUDE_STR2 = "','".join(['vaccine','covid vaccine','pfizer vaccine','moderna vaccine',
    'flu vaccine','mmr vaccine','booster','vaccine injection','mrna covid-19 vaccine','pfizer'])

user_sent_strong = pd.read_sql(f"""
    SELECT tr.user_id,
           AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
                WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_sent,
           COUNT(*) as n_reports
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.signal_strength = 'strong'
    AND t.canonical_name NOT IN ('supplements','medication','treatment','therapy','drug','drugs',
        'vitamin','prescription','pill','pills','dosage','dose',
        '{CAUSAL_EXCLUDE_STR2}')
    GROUP BY tr.user_id
""", conn)
user_sent_strong['group'] = user_sent_strong['user_id'].apply(lambda x: 'POTS' if x in pots_set else 'Non-POTS')

pots_strong = user_sent_strong[user_sent_strong['group'] == 'POTS']['avg_sent']
nonpots_strong = user_sent_strong[user_sent_strong['group'] == 'Non-POTS']['avg_sent']

if len(pots_strong) >= 5 and len(nonpots_strong) >= 10:
    u_ss, p_ss = mannwhitneyu(pots_strong, nonpots_strong, alternative='two-sided')
    r_ss = 1 - (2 * u_ss) / (len(pots_strong) * len(nonpots_strong))

    verdict = ("The POTS disadvantage <strong>persists</strong> when restricted to strong-signal reports, "
               "confirming this is not an artifact of weak or ambiguous sentiment extraction."
               if p_ss < 0.05 else
               "The difference <strong>does not reach significance</strong> in strong-signal-only data, "
               "though the direction is consistent. The smaller sample may be underpowered.")

    display(HTML(f"""
    <div style='background: #f8f9fa; padding: 15px; border-radius: 8px; max-width: 700px;'>
    <h4>Sensitivity Check: Strong-Signal Reports Only</h4>
    <table style='font-size: 13px;'>
    <tr><th></th><th>POTS</th><th>Non-POTS</th></tr>
    <tr><td>Users (strong-signal)</td><td>{len(pots_strong)}</td><td>{len(nonpots_strong)}</td></tr>
    <tr><td>Mean sentiment</td><td>{pots_strong.mean():.3f}</td><td>{nonpots_strong.mean():.3f}</td></tr>
    <tr><td>Median sentiment</td><td>{pots_strong.median():.3f}</td><td>{nonpots_strong.median():.3f}</td></tr>
    <tr><td>Mann-Whitney p</td><td colspan=2>{p_ss:.4f}</td></tr>
    <tr><td>Effect size (r)</td><td colspan=2>{r_ss:.3f}</td></tr>
    </table>
    <p style='margin-top: 10px; font-size: 13px;'><strong>Verdict:</strong> {verdict}</p>
    </div>
    """))
else:
    display(HTML("<p><em>Insufficient strong-signal reports from POTS users for a reliable sensitivity check.</em></p>"))


## 8. Tiered Recommendations

In [ ]:

# ── Tiered recommendations ──
recs = [
    {'tier': 'Strong', 'treatment': 'Electrolytes', 'evidence': '89% positive (n=40, p<0.001 vs 50%)',
     'note': 'Foundational POTS management. Universally recommended, data confirms.'},
    {'tier': 'Strong', 'treatment': 'Propranolol / Beta Blockers', 'evidence': '70-80% positive (n=37 each)',
     'note': 'First-line POTS treatment. Community data consistent with clinical guidelines.'},
    {'tier': 'Strong', 'treatment': 'Ivabradine', 'evidence': '74% positive (n=26, p=0.007 vs 50%)',
     'note': 'Strong support as heart rate control alternative. RECOVER trial data pending.'},
    {'tier': 'Moderate', 'treatment': 'Magnesium', 'evidence': '100% pos among POTS users (n=7), 94% overall (n=56)',
     'note': 'Excellent tolerance. Zero negative reports from POTS patients.'},
    {'tier': 'Moderate', 'treatment': 'Mestinon (pyridostigmine)', 'evidence': '80% positive (n=9)',
     'note': 'Cholinesterase inhibitor for autonomic support. Promising but small sample.'},
    {'tier': 'Moderate', 'treatment': 'Midodrine', 'evidence': '75% positive (n=7)',
     'note': 'Vasopressor for orthostatic hypotension. Standard POTS pharmacotherapy.'},
    {'tier': 'Moderate', 'treatment': 'NAC (N-acetylcysteine)', 'evidence': '86% pos among POTS (n=5) vs 65% non-POTS',
     'note': 'Antioxidant. May benefit POTS patients more than general population.'},
    {'tier': 'Preliminary', 'treatment': 'Nattokinase', 'evidence': '25% pos among POTS (n=7) vs 70% non-POTS',
     'note': 'Caution: POTS patients may tolerate poorly. Monitor for worsening symptoms.'},
    {'tier': 'Preliminary', 'treatment': 'Famotidine', 'evidence': '20% pos among POTS (n=5) vs 68% non-POTS',
     'note': 'H2 blocker. May not help POTS despite high MCAS comorbidity.'},
    {'tier': 'Preliminary', 'treatment': 'Escitalopram', 'evidence': '0% pos among POTS (n=4) vs 71% non-POTS',
     'note': 'Very small sample. Emotional blunting cited as deal-breaker by POTS patients.'},
]

recs_df = pd.DataFrame(recs)
tier_colors = {'Strong': '#2ecc71', 'Moderate': '#f39c12', 'Preliminary': '#e74c3c'}

fig, axes = plt.subplots(3, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 4, 3]})

for ax, tier in zip(axes, ['Strong', 'Moderate', 'Preliminary']):
    tier_data = recs_df[recs_df['tier'] == tier]
    y_pos = range(len(tier_data))

    ax.barh(y_pos, range(len(tier_data), 0, -1), color=tier_colors[tier], alpha=0.3, height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tier_data['treatment'].values, fontsize=10)

    for i, (_, row) in enumerate(tier_data.iterrows()):
        ax.text(0.02, i, row['evidence'], va='center', fontsize=8.5,
                transform=ax.get_yaxis_transform(), style='italic')

    ax.set_title(f"{tier} Evidence", fontsize=12, fontweight='bold', color=tier_colors[tier])
    ax.set_xlim(0, len(tier_data))
    ax.set_xticks([])
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

# HTML table
html_parts = []
for tier in ['Strong', 'Moderate', 'Preliminary']:
    color = tier_colors[tier]
    tier_data = recs_df[recs_df['tier'] == tier]
    html_parts.append(f"<h4 style='color: {color}; margin-top: 15px;'>{tier} Evidence</h4>")
    html_parts.append("<table style='font-size: 13px; width: 100%; border-collapse: collapse;'>")
    html_parts.append("<tr style='background: #f0f0f0;'><th style='text-align:left; padding:6px;'>Treatment</th>"
                      "<th style='text-align:left; padding:6px;'>Evidence</th>"
                      "<th style='text-align:left; padding:6px;'>Note</th></tr>")
    for _, row in tier_data.iterrows():
        html_parts.append(f"<tr><td style='padding:6px; font-weight:bold;'>{row['treatment']}</td>"
                          f"<td style='padding:6px;'>{row['evidence']}</td>"
                          f"<td style='padding:6px;'>{row['note']}</td></tr>")
    html_parts.append("</table>")
display(HTML("".join(html_parts)))


## 9. Conclusion

POTS patients within the Long COVID community are not simply Long COVID patients who also have tachycardia. They represent a distinct subgroup characterized by extreme medical complexity (median 6 co-occurring conditions vs. 2), high treatment-seeking behavior (twice the treatments tried, 3.6x the posting rate), and measurably worse treatment outcomes (mean sentiment 0.27 vs. 0.50). This profile is consistent with clinical descriptions of POTS as a syndrome that frequently overlaps with MCAS, ME/CFS, and EDS, creating a treatment-resistant phenotype.

The most actionable finding is that several treatments beloved by the broader Long COVID community --- nattokinase, famotidine, and escitalopram --- appear to perform poorly for POTS patients specifically. These are preliminary signals from small samples (n=4--7 POTS users per treatment), but the consistency of the pattern is notable: treatments targeting microclots (nattokinase), histamine (famotidine), and serotonin (escitalopram) all underperform, while treatments targeting autonomic dysfunction directly (beta blockers, ivabradine, electrolytes, magnesium) perform well. This suggests that POTS patients may need treatment strategies tailored to their autonomic dysfunction rather than following the general Long COVID playbook.

For a POTS patient asking what to try: start with the established autonomic toolkit (electrolytes, propranolol or ivabradine, magnesium), consider mestinon and midodrine with specialist guidance, and approach community-popular supplements like nattokinase with more caution than the average Long COVID patient would need. SSRIs may help some POTS patients but escitalopram specifically drew negative responses, and the emotional blunting side effect appears to be a particular concern for this population.

What remains unanswered: whether the poor response to nattokinase reflects a genuine biological difference or a selection effect (sicker patients trying more things and rating them lower), whether the MCAS-targeted treatments might work better with longer duration or different dosing in POTS patients, and whether the POTS-specific treatments (ivabradine, mestinon) outperform in controlled comparisons or simply benefit from expectation effects in a population that has tried everything else first.

## 10. Research Limitations

1. **Selection bias:** r/covidlonghaulers users are self-selected. They skew toward English-speaking, internet-literate patients with enough energy to post. POTS patients here may represent the more functional end of the POTS spectrum --- those too ill to post are invisible.

2. **Reporting bias:** Patients with strong opinions (positive or negative) are more likely to write about treatments than those with neutral experiences. This inflates both tails of the sentiment distribution and may exaggerate differences between groups.

3. **Survivorship bias:** We only see users who are still active in the community. Patients who recovered (or deteriorated beyond posting ability) are missing. The POTS group's high comorbidity may partially reflect that sicker patients stay in the community longer.

4. **Recall bias:** Treatment reports reflect how users remember and describe their experiences, not objective clinical outcomes. Sentiment extraction adds another layer of interpretation. A user saying a treatment "didn't help with my main symptoms but helped with sleep" may be coded differently depending on which sentence the pipeline focuses on.

5. **Confounding:** POTS patients try more treatments and have more conditions. Their lower positive rates may reflect disease severity rather than treatment inefficacy. A treatment that works for mild Long COVID may fail for severe Long COVID + POTS + MCAS + ME/CFS regardless of POTS specifically.

6. **No control group:** There is no untreated comparison group. We compare POTS to non-POTS, but both groups are self-treating. We cannot distinguish "this treatment doesn't work for POTS" from "POTS patients rate everything lower due to illness severity."

7. **Sentiment vs. efficacy:** Sentiment extraction from Reddit posts is a proxy for treatment experience, not a measure of clinical efficacy. A positive sentiment may reflect placebo effect, initial improvement before relapse, or social pressure to report positively about popular treatments.

8. **Temporal snapshot:** One month of data (March 11 -- April 10, 2026) captures a snapshot, not a trend. Treatment popularity and community consensus shift over time. These findings may not generalize to other periods.

In [ ]:

display(HTML('<div style="font-size: 1.2em; font-weight: bold; font-style: italic; margin-top: 30px; '
             'padding: 15px; border: 2px solid #e74c3c; border-radius: 8px; text-align: center;">'
             'These findings reflect reporting patterns in online communities, not population-level '
             'treatment effects. This is not medical advice.'
             '</div>'))
